# 1. Header, Citations, Purpose & Topic Bridges

**Topic:** Spiking Neural Networks, LIF, Surrogate Gradients, and Capsules
**Category:** 05_architectures / 07_multimodal_neuro_symbolic_and_neuromorphic

### Citations & Core References
- Gerstner, W., & Kistler, W. M. (2002). *Spiking Neuron Models: Single Neurons, Populations, Plasticity*. Cambridge University Press.
- Neftci, E. O., et al. (2019). *Surrogate Gradient Learning in Spiking Neural Networks*. IEEE Signal Processing Magazine.
- Sabour, S., Frosst, N., & Hinton, G. E. (2017). *Dynamic Routing Between Capsules*. NeurIPS.

## 2. Mathematical Foundations & Governing Equations

### Discrete-Time Leaky Integrate-and-Fire (LIF) Dynamics
$$V[t] = \beta \cdot V[t-1] \odot (1 - S[t-1]) + I[t], \quad S[t] = \Theta(V[t] - V_{\text{th}})$$

### Vector Squashing & Capsule Dynamic Routing
$$v_j = \frac{\|s_j\|^2}{1 + \|s_j\|^2} \frac{s_j}{\|s_j\|_2}, \quad s_j = \sum_i c_{ij} \hat{u}_{j|i}, \quad b_{ij} \leftarrow b_{ij} + \hat{u}_{j|i} \cdot v_j$$

In [ ]:
# 3. Deterministic Setup & Micro-Data Generator
import torch
import torch.nn.functional as F

torch.manual_seed(42)

# LIF setup: T=4 time steps, B=2 batches, D=6 neurons
T, B, D = 4, 2, 6
currents = torch.randn(T, B, D) * 0.6 + 0.4

# Capsule setup: B=2, N_in=4, N_out=3, D_out=4
u_hat = torch.randn(2, 4, 3, 4) * 0.5
print(f"Setup complete: currents {currents.shape}, capsule predictions {u_hat.shape}")

In [ ]:
# 4. First-Principles Implementation & Native Parity
def lif_simulation(currents, beta=0.8, v_th=1.0):
    t_steps, b, d = currents.shape
    v = torch.zeros(b, d, dtype=currents.dtype, device=currents.device)
    s = torch.zeros(b, d, dtype=currents.dtype, device=currents.device)
    all_v, all_s = [], []
    for t in range(t_steps):
        v = beta * v * (1.0 - s) + currents[t]
        s = (v >= v_th).to(currents.dtype)
        all_v.append(v)
        all_s.append(s)
    return torch.stack(all_s, dim=0), torch.stack(all_v, dim=0)

def capsule_squash(s):
    norm_sq = torch.sum(s ** 2, dim=-1, keepdim=True)
    norm = torch.sqrt(norm_sq + 1e-8)
    scale = norm_sq / (1.0 + norm_sq)
    return scale * (s / norm)

def capsule_dynamic_routing(u_hat, num_iters=3):
    b, n_in, n_out, d_out = u_hat.shape
    b_logits = torch.zeros(b, n_in, n_out, dtype=u_hat.dtype, device=u_hat.device)
    v = torch.zeros(b, n_out, d_out, dtype=u_hat.dtype, device=u_hat.device)
    for i in range(num_iters):
        c = F.softmax(b_logits, dim=-1)
        s = torch.sum(c.unsqueeze(-1) * u_hat, dim=1)
        v = capsule_squash(s)
        if i < num_iters - 1:
            agreement = torch.sum(u_hat * v.unsqueeze(1), dim=-1)
            b_logits = b_logits + agreement
    return v

spikes, voltages = lif_simulation(currents, beta=0.8, v_th=1.0)
v_caps = capsule_dynamic_routing(u_hat, num_iters=3)
print(f"LIF spikes: {spikes.shape}, Voltages: {voltages.shape}, Capsules: {v_caps.shape}")

In [ ]:
# 5. Quantitative Invariants & Parity Verification
# Invariant 1: Spikes must be strictly binary
assert torch.all((spikes == 0.0) | (spikes == 1.0)), "Spike values are non-binary"

# Invariant 2: Capsule squash norms must strictly be bounded in [0, 1)
caps_norms = torch.norm(v_caps, p=2, dim=-1)
assert torch.all(caps_norms < 1.0), "Capsule norms exceed 1.0"
assert torch.all(caps_norms >= 0.0), "Capsule norms are negative"
print("All SNN binary invariants and Capsule norm constraints verified.")

In [ ]:
# 6. Performance Profile & Diagnostic Visualizations
import time

iters = 100
t0 = time.perf_counter()
for _ in range(iters):
    _ = lif_simulation(currents, beta=0.8, v_th=1.0)
    _ = capsule_dynamic_routing(u_hat, num_iters=3)
dt = (time.perf_counter() - t0) / iters * 1000.0
print(f"Fused neuromorphic step latency: {dt:.3f} ms / step")

In [ ]:
# 7. Pathological Misuse vs. Corrected Implementation
# Pathological: extremely high threshold results in complete network silence
v_th_extreme = 100.0
spikes_silent, _ = lif_simulation(currents, beta=0.8, v_th=v_th_extreme)
total_spikes = spikes_silent.sum().item()
print(f"Pathological silent network total spikes: {total_spikes} (zero activity)")

# Corrected: Adaptive threshold or threshold calibrated to input current distribution
v_th_calibrated = 1.0
spikes_active, _ = lif_simulation(currents, beta=0.8, v_th=v_th_calibrated)
print(f"Calibrated network total spikes: {spikes_active.sum().item()} (healthy spiking activity)")

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Temporal Sparsity**: LIF neurons compute on an event-driven basis, consuming near-zero power in the absence of sensory stimulus.
2. **Surrogate Differentiation**: Smoothing the backward pass allows end-to-end backpropagation through time across non-differentiable threshold functions.
3. **Part-Whole Equivariance**: Capsule dynamic routing replaces spatial pooling with agreement-based consensus, preserving hierarchical structural relationships.